# Qwen3-TTS Mystery Narration - Google Colab

Voice-cloning TTS for the antique mystery podcast, using Alibaba's Qwen3-TTS (January 2026).

## Key differences from VibeVoice

| Feature | VibeVoice | Qwen3-TTS |
|---|---|---|
| Architecture | Single-pass encoder-decoder | Paragraph-chunked generation |
| EOS truncation | ~10% failure rate | Not possible (chunked) |
| Speaker format | `Speaker N:` labels required | Plain paragraphs |
| Reference transcript | Not required | **Required** (auto-transcribed below) |
| Model size | 5.4 GB | ~4.5 GB |
| Output sample rate | 24,000 Hz | 12,500 Hz |
| Buffer text needed | Yes | No |
| Completeness check | Yes (Whisper) | No (each chunk completes fully) |

No buffer text, no re-runs needed for truncation, no Whisper completeness check.

## Requirements

- Google Colab with GPU runtime (free T4 works; A100 is faster)
- ~6 GB GPU VRAM minimum
- ~10 minutes for setup + generation


## Step 1: Check GPU Availability

In [ ]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✓ GPU detected: {gpu_name}")
    print(f"✓ VRAM: {gpu_memory:.1f} GB")
    if gpu_memory < 6:
        print(f"\n⚠️  WARNING: GPU has only {gpu_memory:.1f}GB VRAM.")
        print("   Minimum 6GB recommended. Try the 0.6B model if generation fails.")
    else:
        print("\n✓ GPU has sufficient VRAM for Qwen3-TTS!")
else:
    print("❌ ERROR: No GPU detected.")
    print("   Go to Runtime > Change runtime type > GPU")
    raise RuntimeError("GPU required for Qwen3-TTS")


## Step 2: Install Qwen3-TTS

Installs the `qwen-tts` package plus audio utilities.
`flash-attn` is optional — it speeds up generation on A100/H100 but is not required on T4.


In [ ]:
%%bash
pip install -q -U qwen-tts
pip install -q soundfile librosa openai-whisper

# Optional: uncomment on A100/H100 for faster generation
# pip install -U flash-attn --no-build-isolation

echo "✓ Installation complete"


## Step 3: Paste Your Mystery Story

Paste plain paragraphs — **no `Speaker N:` labels** (Qwen3-TTS doesn't use them).
Separate paragraphs with a blank line. Each paragraph becomes one generation chunk.

**Punctuation note:** Qwen3-TTS handles Unicode fine, but normalising smart quotes
and em-dashes to ASCII prevents any edge-case issues.


In [ ]:
# Paste your mystery story here as plain paragraphs.
# No 'Speaker 0:' labels. Separate paragraphs with a blank line.

MYSTERY_STORY = (
    "I have always been drawn to objects that carry faint echoes of lives long past - things that seem charged with memory, as if the events they have witnessed are pressed into their very surfaces. Some of these connections are gentle: a letter pressed into the pages of an old book, a chipped teacup that once rested on a kitchen table.\n"
    "\n"
    "Others are darker, tied to moments of violence, misfortune, or sudden loss. Yet even these pieces exert a strange pull.\n"
    "\n"
    "Perhaps it is because objects endure in ways people cannot. They remain long after the events around them have faded, quietly holding fragments of stories that might otherwise be forgotten.\n"
    "\n"
    "I once came across a silver candelabra, six arms twisting upward like delicate fingers, each engraved with intricate floral patterns. Its base was heavy and cool, the kind of silver that has been polished repeatedly over decades.\n"
    "\n"
    "The story attached to it was unsettling: it had belonged to a silver dealer in the midwest, a man whose life ended violently. He had been found murdered in his home, and its twin had been implicated in the tragedy.\n"
    "\n"
    "It is now locked away in evidence permanently, sealed in an archive of photographs, gloves, and numbered tags. One candelabra in circulation, the other forever preserved in the cold bureaucracy of the law. That duality made the one I held irresistible - not merely craftsmanship or history, but a fragment of a story only partially recoverable, an object that had both survived and outlived its context.\n"
    "\n"
    "Objects connected to crimes or deaths rarely travel into the world of collectors in straightforward ways. Investigators often seize items, catalog them, and store them in evidence rooms.\n"
    "\n"
    "Some items remain there for decades, particularly in serious cases like homicide, while others may be destroyed, returned, or otherwise removed from custody once legal requirements are satisfied. Many objects, however, never enter evidence at all.\n"
    "\n"
    "Perhaps they are overlooked, deemed irrelevant, or were present in a room but played no identifiable role in the events. A candlestick left on a sideboard, a clock ticking quietly in the corner, a chair pushed back from a table - all might have borne silent witness to violence without ever being touched by investigators. In such cases, they pass into circulation almost anonymously, eventually showing up in estate sales, antique shops, and private collections, carrying stories only fragments of which remain.\n"
    "\n"
    "That uncertainty is part of their fascination. Objects like these are more than their materials.\n"
    "\n"
    "They become vessels for human decisions, impulses, and consequences. The appeal rarely lies in the tragedy itself, but in the sense that these objects endured it, that they persisted while lives around them ended, shifted, or fractured.\n"
    "\n"
    "Holding such a piece creates an unexpected intimacy with the past. One begins to wonder how many hands have lifted it, polished it, or moved it from place to place.\n"
    "\n"
    "Perhaps it was present during an argument that escalated to violence, or it sat unnoticed while something irreversible unfolded nearby. The truth is rarely recoverable in full, yet the possibility of hidden stories gives the object depth, a weight that extends beyond its age or aesthetic beauty.\n"
    "\n"
    "For many collectors, this sense of continuity is the real attraction. Human life is fleeting, but material objects endure.\n"
    "\n"
    "Silver withstands generations of polishing, wood darkens without forgetting the marks left upon it. Even the smallest imperfections - a dent, a scratch, a chip - can suggest moments that passed unrecorded. In this way, objects serve as bridges between eras, allowing us to imagine lives we cannot fully know and moments of fear, passion, or desperation that unfolded in rooms now long changed.\n"
    "\n"
    "And now, in my studio, the candelabra rests on my piano, bathed in the pale silver of moonlight filtering through the window. Its polished arms catch the glow, reflecting not only the light but the weight of the life it has witnessed.\n"
    "\n"
    "I find myself pondering the fate of its twin - the one sealed in evidence, never to be held, moved, or admired. How different might it have been to see it here, quiet and luminous, rather than locked behind glass and numbered tags?\n"
    "\n"
    "The comparison deepens the allure of the candelabra before me, emphasizing the strange fortune that allowed it to survive, to endure, to reflect the moonlight in a way its companion never will. In this moment, I feel a peculiar intimacy with both the object and the past it embodies.\n"
    "\n"
    "I imagine the rooms it has been in, the hands that have held it, the life it witnessed, and the alternative histories it might have shared. There is solace here, a quiet reminder that some traces of the past can be touched, held, and observed, and that in their endurance, they keep stories alive that might otherwise have been lost to time.\n"
    "\n"
)

MYSTERY_STORY = MYSTERY_STORY.strip()

# Normalise punctuation (Qwen3-TTS handles Unicode well, but ASCII is safest)
MYSTERY_STORY = MYSTERY_STORY.replace("\u2019", "'").replace("\u2018", "'")
MYSTERY_STORY = MYSTERY_STORY.replace("\u201c", '"').replace("\u201d", '"')
MYSTERY_STORY = MYSTERY_STORY.replace("\u2014", " - ").replace("\u2013", "-")
MYSTERY_STORY = MYSTERY_STORY.replace("\u2026", "...")

# Split into paragraph chunks (one per blank line)
CHUNKS = [p.strip() for p in MYSTERY_STORY.split("\n\n") if p.strip()]

word_count = len(MYSTERY_STORY.split())
print(f"Story: {word_count} words across {len(CHUNKS)} paragraph chunks")
print(f"Estimated audio: ~{word_count / 150:.1f} minutes")
print(f"Estimated generation time: ~{word_count / 150 * 4:.0f} minutes")
print(f"\nChunk lengths: {[len(c.split()) for c in CHUNKS]} words")


## Step 4: Load Voice Reference and Auto-Transcribe

Qwen3-TTS requires **both** a reference audio file **and** a text transcript of that audio.
The transcript teaches the model which voice characteristics to clone.

The cell below:
1. Loads your `EdisonsGhostPhone.mp3` from Google Drive
2. Trims it to 30 seconds (more than enough for cloning)
3. Saves the trimmed clip as a temp WAV
4. Auto-transcribes it with Whisper (runs on CPU, ~30 seconds)

**⚠️ Verify the transcript before proceeding.** Errors in `REF_TEXT` reduce voice clone quality.
You can correct it manually in the `REF_TEXT = ...` line that is printed.


In [ ]:
import librosa
import soundfile as sf
import whisper
import tempfile, os
import numpy as np
from pathlib import Path

# Load voice reference from Google Drive
from google.colab import drive
drive.mount('/content/drive')
VOICE_REFERENCE_PATH = "/content/drive/MyDrive/EdisonsGhostPhone.mp3"

if not Path(VOICE_REFERENCE_PATH).exists():
    raise FileNotFoundError(
        f"Voice reference not found: {VOICE_REFERENCE_PATH}\n"
        "Upload EdisonsGhostPhone.mp3 to Google Drive root and re-run."
    )

# Load and trim to first 30 seconds
CLONE_SR = 24000
raw_audio, raw_sr = librosa.load(VOICE_REFERENCE_PATH, sr=CLONE_SR, mono=True)
duration = len(raw_audio) / CLONE_SR
print(f"✓ Loaded reference audio: {duration:.1f} seconds")

MAX_REF_SEC = 30
if duration > MAX_REF_SEC:
    raw_audio = raw_audio[:int(MAX_REF_SEC * CLONE_SR)]
    print(f"✓ Trimmed to first {MAX_REF_SEC} seconds")

# Save trimmed clip as temp WAV for Qwen3-TTS and Whisper
REF_WAV_PATH = "/content/reference_trimmed.wav"
sf.write(REF_WAV_PATH, raw_audio, CLONE_SR)
print(f"✓ Saved trimmed reference to: {REF_WAV_PATH}")

# Auto-transcribe with Whisper (base model, CPU)
print("\nTranscribing reference audio with Whisper (CPU, ~30s)...")
w_model = whisper.load_model("base")
result = w_model.transcribe(REF_WAV_PATH, language="en", fp16=False)
REF_TEXT = result["text"].strip()

print(f"\n✓ Auto-transcribed reference text:")
print(f'  REF_TEXT = "{REF_TEXT}"')
print("\n⚠️  Verify this is accurate. Edit REF_TEXT below if needed.")
print("   Transcript accuracy directly affects voice clone quality.")

# -- MANUAL OVERRIDE (uncomment and edit if Whisper got it wrong) --
# REF_TEXT = "Your manually corrected transcript here."


## Step 5: Load Qwen3-TTS Model

Downloads and loads the 1.7B voice-cloning model (~4.5 GB, takes 2-3 minutes first run).

**Model options:**
- `Qwen3-TTS-12Hz-1.7B-Base` — recommended, best quality
- `Qwen3-TTS-12Hz-0.6B-Base` — faster, lower VRAM (use if T4 runs out of memory)


In [ ]:
import torch
from qwen_tts import Qwen3TTSModel

MODEL_ID = "Qwen/Qwen3-TTS-12Hz-1.7B-Base"
# MODEL_ID = "Qwen/Qwen3-TTS-12Hz-0.6B-Base"  # Uncomment for lower-VRAM fallback

print(f"Loading {MODEL_ID} (~4.5 GB download)...")
print("This may take 2-3 minutes on first run...")

model = Qwen3TTSModel.from_pretrained(
    MODEL_ID,
    device_map="cuda:0",
    dtype=torch.bfloat16,
    # attn_implementation="flash_attention_2"  # Uncomment if flash-attn installed
)

print(f"\n✓ Qwen3-TTS model loaded on GPU")
print(f"✓ Model: {MODEL_ID}")


## Step 6: Generate Mystery Narration

Each paragraph is generated independently, then stitched together with a short pause.
Because each chunk completes fully, there is **no EOS truncation** and no re-runs needed.

**Tuning parameters:**
- `TEMPERATURE` — 0.7–0.9 controls expressiveness. 0.8 is a good default; avoid 0.9 (can sound robotic).
- `PAUSE_BETWEEN_CHUNKS_MS` — silence inserted between paragraphs (300–600 ms typical).


In [ ]:
import numpy as np
import soundfile as sf
from IPython.display import Audio, display

# Generation parameters
LANGUAGE = "English"
TEMPERATURE = 0.8
TOP_P = 0.9
PAUSE_BETWEEN_CHUNKS_MS = 400  # ms of silence between paragraphs

print("=" * 60)
print("GENERATING AUDIO")
print("=" * 60)
print(f"Model          : {MODEL_ID}")
print(f"Chunks         : {len(CHUNKS)}")
print(f"Language       : {LANGUAGE}")
print(f"Temperature    : {TEMPERATURE}")
print(f"Pause between  : {PAUSE_BETWEEN_CHUNKS_MS} ms")
print(f"\nGenerating... (approx {len(MYSTERY_STORY.split()) / 150 * 4:.0f} min total)")
print("\u2615 Go get coffee! \u2615\n")

audio_chunks = []
sample_rate = None

for i, chunk in enumerate(CHUNKS):
    word_count_chunk = len(chunk.split())
    print(f"Chunk {i+1:2d}/{len(CHUNKS)}  ({word_count_chunk:3d} words)  {chunk[:60]}...")
    wavs, sr = model.generate_voice_clone(
        text=chunk,
        language=LANGUAGE,
        ref_audio=REF_WAV_PATH,
        ref_text=REF_TEXT,
        temperature=TEMPERATURE,
        top_p=TOP_P,
        max_new_tokens=2048,
    )
    audio_chunks.append(wavs[0])
    if sample_rate is None:
        sample_rate = sr

print(f"\n✓ All {len(CHUNKS)} chunks generated. Sample rate: {sample_rate} Hz")

# Stitch chunks with silence between paragraphs
pause_samples = int(PAUSE_BETWEEN_CHUNKS_MS / 1000 * sample_rate)
silence = np.zeros(pause_samples, dtype=np.float32)

parts = []
for i, chunk_audio in enumerate(audio_chunks):
    parts.append(chunk_audio.astype(np.float32))
    if i < len(audio_chunks) - 1:
        parts.append(silence)

audio_array = np.concatenate(parts)
actual_duration = len(audio_array) / sample_rate / 60

print("\n" + "=" * 60)
print("\u2713 GENERATION COMPLETE!")
print("=" * 60)
print(f"\u2713 Audio duration : {actual_duration:.2f} minutes")
print(f"\u2713 Sample rate    : {sample_rate} Hz")
print(f"\u2713 Audio samples  : {len(audio_array):,}")

output_path = "/content/mystery_narration.wav"
sf.write(output_path, audio_array, sample_rate)
print(f"\u2713 Saved to       : {output_path}")

print("\n\U0001f3a7 Listen to your mystery narration:")
display(Audio(audio_array, rate=sample_rate))


## Step 6.5: Add Ending Snippet (Optional)

Appends `ending.mp3` from Google Drive with a configurable silent gap.
This step is identical to the VibeVoice notebook.


In [ ]:
from pathlib import Path

ENDING_PATH = "/content/drive/MyDrive/ending.mp3"

if ENDING_PATH and Path(ENDING_PATH).exists():
    print(f"✓ Ending file found: {ENDING_PATH}")
    ENDING_ENABLED = True
else:
    print("ℹ️  No ending file found. Skipping ending snippet.")
    ENDING_ENABLED = False


In [ ]:
if ENDING_ENABLED:
    print("Installing ffmpeg...")
    import subprocess
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg"], check=True)
    print("✓ ffmpeg installed")

    try:
        from pydub import AudioSegment
    except ImportError:
        import subprocess
        subprocess.run(["pip", "install", "-q", "pydub"])
        from pydub import AudioSegment

    print("=" * 60)
    print("COMBINING AUDIO WITH ENDING")
    print("=" * 60)

    SILENT_GAP_MS = 1500   # Silence between narration and ending (ms)
    CROSSFADE_MS  = 0      # Crossfade duration (0 = clean cut)

    narration = AudioSegment.from_wav("/content/mystery_narration.wav")
    print(f"✓ Loaded narration : {len(narration) / 1000:.1f}s")

    ending = AudioSegment.from_file(ENDING_PATH)
    print(f"✓ Loaded ending    : {len(ending) / 1000:.1f}s")

    combined = narration + AudioSegment.silent(duration=SILENT_GAP_MS)
    if CROSSFADE_MS > 0:
        combined = combined.append(ending, crossfade=CROSSFADE_MS)
    else:
        combined = combined + ending

    final_output_path = "/content/mystery_podcast_final.mp3"
    combined.export(final_output_path, format="mp3", bitrate="192k")
    total = len(combined) / 1000 / 60
    print(f"✓ Combined audio   : {total:.2f} min")
    print(f"✓ Saved to         : {final_output_path}")

    print("\n\U0001f3a7 Listen to the final podcast with ending:")
    display(Audio(final_output_path))

if not ENDING_ENABLED:
    print("ℹ️  Skipping ending combination (no valid ending file)")


## Step 7: Download Your Audio

In [ ]:
from google.colab import files

if ENDING_ENABLED:
    files.download('/content/mystery_podcast_final.mp3')
    print("✓ Download started: mystery_podcast_final.mp3")
else:
    files.download('/content/mystery_narration.wav')
    print("✓ Download started: mystery_narration.wav")


## Tips for Best Results

### Voice Reference Quality

- **Length:** 10–30 seconds produces the best cloning (3 seconds works but is noisier)
- **Clarity:** Clean audio, no background noise, natural speaking pace
- **Transcript accuracy:** `REF_TEXT` must closely match what is spoken in the reference clip.
  Even small errors reduce clone fidelity. Use Whisper auto-transcription as a starting point,
  then correct manually if needed.

### Story Text Format

- Plain paragraphs, separated by a blank line
- No `Speaker N:` labels
- Each paragraph becomes one generation chunk — keep paragraphs to 2–4 sentences each
  for best prosody. Very long paragraphs (>100 words) may be split internally.

### Generation Parameters

- **`TEMPERATURE = 0.8`** — recommended. Avoid 0.9 (can produce robotic tone).
- **`TOP_P = 0.9`** — wider intonation variety.
- **`PAUSE_BETWEEN_CHUNKS_MS = 400`** — adjust for faster/slower pacing between paragraphs.

### Model Variants

| Model | VRAM | Speed | Quality |
|---|---|---|---|
| `Qwen3-TTS-12Hz-1.7B-Base` | ~6 GB | Normal | Best (recommended) |
| `Qwen3-TTS-12Hz-0.6B-Base` | ~4 GB | Faster | Good |

### If Voice Quality is Poor

1. Check that `REF_TEXT` exactly matches the reference audio
2. Try `x_vector_only_mode=True` in `generate_voice_clone()` — skips transcript matching,
   lower quality but doesn't fail on bad transcripts
3. Use a longer reference clip (15–30 seconds)
4. Try `TEMPERATURE = 0.75` for a more consistent/neutral tone

### Troubleshooting

**`CUDA out of memory`**
- Switch to the 0.6B model: change `MODEL_ID` in Step 5
- Reduce chunk size: split long paragraphs manually

**Generation sounds wrong / foreign language**
- Confirm `LANGUAGE = "English"` is set in Step 6
- Check that story text has no garbled characters

**Voice doesn't sound like reference**
- Transcription mismatch is the most common cause — correct `REF_TEXT` manually
- Try a different (cleaner) section of the reference audio

---

**Notebook created for:** Antique Mystery Podcast Generator
**Model:** Qwen3-TTS by Alibaba QwenLM (Apache 2.0)
**GitHub:** https://github.com/QwenLM/Qwen3-TTS
**Switched from:** VibeVoice (Microsoft) — see `docs/TTS_ALTERNATIVES.md` for why
